# Synthetic Dataset Generator

## Run locally version

* Describe a business scenario (e.g. restaurant reviews, support tickets, product catalog); 
* The LLM generates structured synthetic data (CSV or JSON). 
* Runs locally with **OpenRouter** (or OpenAI); **Gradio UI** to configure scenario, number of rows, and format.

Google Colabs [alternative](https://colab.research.google.com/drive/1QsIr7QX2DQmH4_u1W5bU_v0R2ySWhlRY?usp=sharing).

### Packages Install 

In [ ]:
#!pip install transformers==4.45.2 accelerate==0.34.2 huggingface-hub==0.24.0
#!pip install --upgrade transformers accelerate huggingface-hub langchain-huggingface gradio
!pip install "transformers>=4.45.0,<5.0.0" "huggingface-hub>=0.33.5" "accelerate>=0.34.0" "sentence-transformers"

### Setup

In [ ]:
import os
import re
import json
from datetime import datetime as dt
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig, TextIteratorStreamer
import torch


# environment & API client (OpenRouter preferred, fallback OpenAI)
load_dotenv(override=True)

# OpenRouter
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
if openrouter_api_key:
    print("OpenRouter key looks good so far")
else:
    print("OpenRouter key is not set")

client = OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1")
MODEL = "openai/gpt-4o-mini"

# HuggingFace
hf_token = os.getenv('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
    print("HF key looks good so far")
else:
    print("HF key is not set")
login(hf_token, add_to_git_credential=True)

# System prompt: synthetic data only, no commentary, no real PII
SYSTEM_PROMPT = """You are a synthetic dataset generator. Your only job is to output structured data.

Rules:
- Output ONLY the requested format (CSV or JSON). No explanations, no markdown code fences, no extra text.
- For CSV: first line is the header row, then one row per record. Use commas; escape quotes inside fields.
- For JSON: output a single JSON array of objects. Each object is one record with consistent keys.
- Generate realistic but fake data. No real names, emails, or identifiable information.
- Infer a sensible schema from the user's scenario (e.g. for "restaurant reviews" use: reviewer_name, rating, review_text, date).
- Generate exactly the number of records requested.
"""

# HF_TOKEN = os.environ.get("HF_TOKEN", None)
DEFAULT_MODEL = "openai/gpt-4o-mini"

AVAILABLE_MODELS = [
    "openai/gpt-4o-mini",
    "google/gemini-2.5-flash-lite",
    "google/gemini-2.5-flash-image",  	
    "openai/gpt-oss-120b",
    "google/gemma-4-26b-a4b-it:free",
    "google/gemma-4-31b-it:free",
    "meta-llama/Llama-3.2-3B-Instruct"
]

model_map = {
    "openai/gpt-4o-mini": "OPENROUTER",
    "google/gemini-2.5-flash-lite": "OPENROUTER",
    "google/gemini-2.5-flash-image": "OPENROUTER",  	
    "openai/gpt-oss-120b": "OPENROUTER",
    "google/gemma-4-26b-a4b-it:free": "OPENROUTER",
    "google/gemma-4-31b-it:free": "OPENROUTER",
    "meta-llama/Llama-3.2-3B-Instruct": "HF"
}

OpenRouter key looks good so far
HF key looks good so far


python(15817) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(15818) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(15823) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### Functions and Helper Methods

In [15]:
def hf_generate_dataset(model_id: str, messages: dict):
    if torch.cuda.is_available():
        device = "cuda"
    elif torch.backends.mps.is_available():
        device = "mps"
    else:
        device = "cpu"
    
    print(f"Using device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(device)
    streamer = TextStreamer(tokenizer, chat_template="{|file|}")

    kwargs = {
        "trust_remote_code": True,
        "device_map": {"": device} 
    }

    if device == "cuda":
        # quantization_config only works on CUDA. 
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4"
        )
        # Add your quantization config here if you are on NVIDIA
        kwargs["quantization_config"] = quant_config 
    elif device == "mps":
        # Apple Silicon handles memory better with float16
        kwargs["dtype"] = torch.float16
    else:
        # CPU fallback
        kwargs["dtype"] = torch.float32

    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    
    outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)
    return tokenizer.decode(outputs[0])

def generate_dataset(scenario: str, model: str, num_rows: int, output_format: str) -> str:
    """Call the LLM to generate synthetic data. Returns raw CSV or JSON string."""
    if not scenario or not scenario.strip():
        return "Please describe the dataset characteristcs (e.g. 'restaurant reviews with rating and date')."
    
    num_rows = max(1, min(int(num_rows), 50))
    fmt = "CSV" if "csv" in output_format.lower() else "JSON"
    
    user_msg = (
        f"Generate a synthetic dataset with exactly {num_rows} records. "
        f"Scenario: {scenario.strip()}. "
        f"Output format: {fmt} only, no other text."
    )
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg}
    ]
    
    try:
        if model_map[model] == "HF":
            raw = hf_generate_dataset(model, messages)
        else:
            response = client.chat.completions.create(model=model, messages=messages, temperature=0.7)
            raw = (response.choices[0].message.content or "").strip()
        
        # 1. Strip markdown code blocks if the model added them
        if raw.startswith("```"):
            raw = re.sub(r"^```\w*\n", "", raw)
            raw = re.sub(r"\n```\s*$", "", raw)
            raw = raw.strip()

        # 2. RESTORE JSON FORMATTING
        # If the requested format was JSON, we parse and re-indent it
        if fmt == "JSON":
            try:
                # This converts the raw string into a Python list/dictionary
                json_obj = json.loads(raw) 
                # This converts it back to a string with 4-space indentation (Pretty Print)
                return json.dumps(json_obj, indent=4)
            except json.JSONDecodeError:
                # If the LLM returned invalid JSON, we return the raw string 
                # so the user can see what went wrong.
                return raw
        
        # If it's CSV, just return the stripped raw text
        return raw

    except Exception as e:
        return f"🚨 Error: {e}"


def saveData(data, output_format)->str:
    # 1. Create the folder if it doesn't exist
    folder = "Dataset"
    if not os.path.exists(folder):
        os.makedirs(folder)
    
    # 2. Clean the data 
    # LLMs often wrap output in ```csv ... ``` or ```json ... ```
    # This removes those markers so the file is clean
    clean_data = data.replace("```csv", "").replace("```json", "").replace("```", "").strip()

    saved_time = int(dt.today().timestamp())
    
    # 3. Define the filename based on the format
    filename = f"{folder}/dataset_{saved_time}.{output_format.lower()}"    
    try:
        # 4. Write the file
        with open(filename, "w", encoding="utf-8") as f:
            # If it's JSON, we should validate it's actually JSON first
            if output_format == "JSON":
                try:
                    # Try to parse it to make sure it's valid JSON
                    json_obj = json.loads(clean_data)
                    # Write it back with nice indentation (4 spaces)
                    f.write(json.dumps(json_obj, indent=4))
                except:
                    # If it's not valid JSON, just write it as raw text
                    f.write(clean_data)
            else:
                # For CSV, just write the cleaned text directly
                f.write(clean_data)
                
        return "✅ File successfully saved to the Dataset folder!"
    except Exception as e:
        return f"🚨 Error saving file: {str(e)}"


### Gradio App

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown(
        """
        ## 🌌 Synthetic Dataset Generator powered by Open-Source LLMs
        * Give a simple or a more detailed description of the data you want, e.g.:
            - Product Catalog with product_id as integer (unique), product_name as String(30), price as Double, category_cod as String(3) upercase not null, brand_cod as String(3) upercase not null.
            - Customer Support Tickets with id, subject, status
        * Choose number of rows and output format. 
        * Choose the output format: CSV or JSON.        
        """
    )
    
    with gr.Row():
        scenario = gr.Textbox(
            label="✍️ Dataset description",
            placeholder="e.g. Restaurant reviews with reviewer_name, rating (1-5), review_text, date",
            lines=5
        )
        gr.Markdown("""
        **Tips:**
        - Be specific about data types and value ranges
        - Mention relationships between fields
        - Specify any domain (medical, finance, e-commerce...)
        - Add constraints like date ranges or enums
        """)
        
    with gr.Row():
        model = gr.Dropdown(
            choices=AVAILABLE_MODELS, 
            value=DEFAULT_MODEL, 
            label="🧠 LLM Model"
        )
        num_rows = gr.Slider(1, 50, value=5, step=1, label="🔢 Number of rows")
        output_format = gr.Radio(["JSON", "CSV"], value="CSV", label="⚙️ Output Format")
        
    btn = gr.Button("🔄 Generate", variant="primary")
    out = gr.Textbox(label="📄 Generated data", lines=12)
    
    # Save button starts disabled and with the text "Save"
    save = gr.Button("🗄️ Save", variant="primary", interactive=False)
    
    # Status label for feedback
    status = gr.Markdown("") 

    # --- BUTTON LOGIC ---

    # 1. GENERATE CLICK
    btn.click(
        fn=generate_dataset, 
        inputs=[scenario, model, num_rows, output_format], 
        outputs=out
    ).then(
        # IMPORTANT FIX: 
        # We update BOTH the interactive state AND the label back to "Save"
        # and we clear the status message.
        fn=lambda: (gr.update(interactive=True, value="🗄️ Save"), ""), 
        outputs=[save, status] 
    )

    # 2. SAVE CLICK
    save.click(
        fn=saveData, 
        inputs=[out, output_format], 
        outputs=status
    ).then(
        # Change button text to indicate completion
        fn=lambda: gr.update(value="✅ Your data has been saved!"), 
        outputs=[save]
    )


demo.launch(share=True, debug=False,inbrowser=True) # , theme=gr.themes.Soft())

* Running on local URL:  http://127.0.0.1:7860


python(15853) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


* Running on public URL: https://9083030bceb49296b0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


python(15854) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Using device: mps


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 10 May 2026

You are a synthetic dataset generator. Your only job is to output structured data.

Rules:
- Output ONLY the requested format (CSV or JSON). No explanations, no markdown code fences, no extra text.
- For CSV: first line is the header row, then one row per record. Use commas; escape quotes inside fields.
- For JSON: output a single JSON array of objects. Each object is one record with consistent keys.
- Generate realistic but fake data. No real names, emails, or identifiable information.
- Infer a sensible schema from the user's scenario (e.g. for "restaurant reviews" use: reviewer_name, rating, review_text, date).
- Generate exactly the number of records requested.<|eot_id|><|start_header_id|>user<|end_header_id|>

Generate a synthetic dataset with exactly 5 records. Scenario: Movie reviews with review_id as integer (unique, primary key), movie_name as string(30) 

### Dataset description and Close Gradio App

In [ ]:
gr.close_all()

# Dataset description suggestion
"""
Movie reviews with review_id as integer (unique, primary key), movie_name as string(30) not null, cinema_cod as char(5) not null,  days_watched_after_release as integer, reviewer_name as string(20) not null, reviewer_media as String(5) not null, review_text as string(200), starts_rating integer range 1 to 5 not null, sentiment_analyses as String(10) the sentiment analyses from the review_text, date_time_review
"""

Closing server running on port: 7860


'\nMovie reviews with review_id as integer (unique, primary key), movie_name as string(30) not null, cinema_cod as char(5) not null,  days_watched_after_release as integer, reviewer_name as string(20) not null, reviewer_media as String(5) not null, review_text as string(200), starts_rating integer range 1 to 5 not null, sentiment_analyses as String(10) the sentiment analyses from the review_text, date_time_review\n'